# AI活用のためのデータベース準備 ハンズオン（Section 08）

このノートブックでは、OpenAI API の準備から、  
Python で呼び出し、共通関数として整理する流れを体験します。

---

## ここでやること
- 前の章で作成したCSVファイルを読み込む
- SQLiteデータベースを作成する
- テーブル（meetings, speeches / qa）を作成する
- 会議、発言データをデータベースに登録する
- 登録されたデータを確認する
- 次の章で使うための土台を整える

In [ ]:
########################################
# テーブル作成
########################################
import sqlite3

conn = None

try:
    conn = sqlite3.connect("ank.db")
    cursor = conn.cursor()

    print("データベース 'ank.db' を作成しました。")

    # meetings テーブル
    sql_create_meetings_table = """
    CREATE TABLE IF NOT EXISTS meetings (
        issueID          TEXT PRIMARY KEY,
        nameOfHouse      TEXT,
        nameOfMeeting    TEXT,
        issue            TEXT,
        date             TEXT
    );
    """
    cursor.execute(sql_create_meetings_table)
    print("テーブル 'meetings' を作成しました。")

    # speeches テーブル
    sql_create_speeches_table = """
    CREATE TABLE IF NOT EXISTS speeches (
        speechID         TEXT PRIMARY KEY,
        issueID          TEXT NOT NULL,
        speechOrder      INTEGER,
        speaker          TEXT,
        speech           TEXT NOT NULL
    );
    """
    cursor.execute(sql_create_speeches_table)
    print("テーブル 'speeches' を作成しました。")

    # qa テーブル
    sql_create_qa_table = """
    CREATE TABLE IF NOT EXISTS qa (
        qa_id            INTEGER PRIMARY KEY AUTOINCREMENT,
        speechID         TEXT NOT NULL,
        question         TEXT NOT NULL,
        answer           TEXT NOT NULL,
        created_at       TEXT DEFAULT CURRENT_TIMESTAMP
    );
    """
    cursor.execute(sql_create_qa_table)
    print("テーブル 'qa' を作成しました。")

    conn.commit()
    print("テーブル作成を保存しました。")

except sqlite3.Error as e:
    print(f"データベース操作中にエラーが発生しました: {e}")
    if conn:
        conn.rollback()

except Exception as e:
    print(f"予期しないエラーが発生しました: {e}")

finally:
    if conn:
        conn.close()
        print("データベース接続をクローズしました。")

In [ ]:
########################################
# Meeting.csv をアップロードして
# meetings テーブルに登録
########################################
import sqlite3
import pandas as pd
from google.colab import files

conn = None

try:
    # -----------------------------
    # ① CSVファイルをアップロード
    # -----------------------------
    uploaded = files.upload()
    csv_file = list(uploaded.keys())[0]

    print(f"アップロードしたファイル: {csv_file}")

    # -----------------------------
    # ② CSVを読み込み
    # -----------------------------
    df_meetings = pd.read_csv(csv_file)

    print("CSVを読み込みました。")
    display(df_meetings.head())

    # -----------------------------
    # ③ データベースに接続
    # -----------------------------
    conn = sqlite3.connect("ank.db")
    cursor = conn.cursor()

    added_count = 0
    skipped_count = 0

    # -----------------------------
    # ④ meetings テーブルに登録
    # -----------------------------
    for _, row in df_meetings.iterrows():
        try:
            cursor.execute("""
                INSERT INTO meetings (
                    issueID,
                    nameOfHouse,
                    nameOfMeeting,
                    issue,
                    date
                ) VALUES (?, ?, ?, ?, ?)
            """, (
                str(row["issueID"]) if pd.notna(row["issueID"]) else None,
                str(row["nameOfHouse"]) if pd.notna(row["nameOfHouse"]) else None,
                str(row["nameOfMeeting"]) if pd.notna(row["nameOfMeeting"]) else None,
                str(row["issue"]) if pd.notna(row["issue"]) else None,
                str(row["date"]) if pd.notna(row["date"]) else None
            ))
            added_count += 1

        except sqlite3.IntegrityError:
            skipped_count += 1

    # -----------------------------
    # ⑤ 保存
    # -----------------------------
    conn.commit()

    print("meetings テーブルへの登録が完了しました。")
    print(f"CSV件数: {len(df_meetings)}")
    print(f"追加件数: {added_count}")
    print(f"スキップ件数: {skipped_count}")

except sqlite3.Error as e:
    print(f"データベース操作中にエラーが発生しました: {e}")
    if conn:
        conn.rollback()

except Exception as e:
    print(f"予期しないエラーが発生しました: {e}")

finally:
    if conn:
        conn.close()
        print("データベース接続をクローズしました。")

In [ ]:
########################################
# Speech.csv をアップロードして
# speeches テーブルに登録
########################################
import sqlite3
import pandas as pd
from google.colab import files

conn = None

try:
    # -----------------------------
    # ① CSVファイルをアップロード
    # -----------------------------
    uploaded = files.upload()
    csv_file = list(uploaded.keys())[0]

    print(f"アップロードしたファイル: {csv_file}")

    # -----------------------------
    # ② CSVを読み込み
    # -----------------------------
    df_speeches = pd.read_csv(csv_file)

    print("CSVを読み込みました。")
    display(df_speeches.head())

    # -----------------------------
    # ③ データベース接続
    # -----------------------------
    conn = sqlite3.connect("ank.db")
    cursor = conn.cursor()

    added_count = 0
    skipped_count = 0

    # -----------------------------
    # ④ speeches テーブルへ登録
    # -----------------------------
    for _, row in df_speeches.iterrows():
        try:
            cursor.execute("""
                INSERT INTO speeches (
                    speechID,
                    issueID,
                    speechOrder,
                    speaker,
                    speech
                ) VALUES (?, ?, ?, ?, ?)
            """, (
                str(row["speechID"]) if pd.notna(row["speechID"]) else None,
                str(row["issueID"]) if pd.notna(row["issueID"]) else None,
                int(row["speechOrder"]) if pd.notna(row["speechOrder"]) else None,
                str(row["speaker"]) if pd.notna(row["speaker"]) else None,
                str(row["speech"]) if pd.notna(row["speech"]) else ""
            ))

            added_count += 1

        except sqlite3.IntegrityError:
            skipped_count += 1

    # -----------------------------
    # ⑤ 保存
    # -----------------------------
    conn.commit()

    print("speeches テーブルへの登録が完了しました。")
    print(f"CSV件数: {len(df_speeches)}")
    print(f"追加件数: {added_count}")
    print(f"スキップ件数: {skipped_count}")

except sqlite3.Error as e:
    print(f"データベース操作中にエラーが発生しました: {e}")
    if conn:
        conn.rollback()

except Exception as e:
    print(f"予期しないエラーが発生しました: {e}")

finally:
    if conn:
        conn.close()
        print("データベース接続をクローズしました。")